# Captura de imágenes
Libreta usada exclusivamente para capturar imagenes bloqueando el rostro de las personas en video

In [4]:
import cv2
import time
from pathlib import Path
from datetime import datetime

# Configuración
CAMERA_INDEX = 1
COOLDOWN_SECONDS = 5.0
OUTPUT_DIR = Path("raw")

# Por precaución, no guardar una captura si el detector no encuentra
# ningún rostro. Esto no garantiza que haya detectado todos los rostros.
SAVE_ONLY_IF_FACE_DETECTED = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)
if face_cascade.empty():
    raise RuntimeError("No se pudo cargar el detector de rostros de OpenCV.")

cap = cv2.VideoCapture(CAMERA_INDEX)
# Configurar resolución deseada
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)  # Ancho
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080) # Alto

if not cap.isOpened():
    raise RuntimeError("No se pudo abrir la webcam.")

last_attempt = time.monotonic() - COOLDOWN_SECONDS
status = "Esperando captura..."

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            print("No se pudo leer un fotograma de la webcam.")
            break

        height, width = frame.shape[:2]

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30, 30)
        )

        # Crear la imagen anonimizada; no se guarda el fotograma original.
        anonymized = frame.copy()

        for (x, y, w, h) in faces:
            # Ampliar la máscara alrededor del rostro detectado.
            margin_x = int(0.25 * w)
            margin_y = int(0.25 * h)

            x1 = max(0, x - margin_x)
            y1 = max(0, y - margin_y)
            x2 = min(width, x + w + margin_x)
            y2 = min(height, y + h + margin_y)

            # Rectángulo negro opaco.
            cv2.rectangle(
                anonymized,
                (x1, y1),
                (x2, y2),
                (0, 0, 0),
                thickness=-1
            )

        now = time.monotonic()

        # Intentar guardar una nueva captura al cumplirse el intervalo.
        if now - last_attempt >= COOLDOWN_SECONDS:
            last_attempt = now

            if len(faces) > 0 or not SAVE_ONLY_IF_FACE_DETECTED:
                filename = datetime.now().strftime(
                    "captura_%Y%m%d_%H%M%S_%f.jpg"
                )
                output_path = OUTPUT_DIR / filename

                saved = cv2.imwrite(
                    str(output_path),
                    anonymized,
                    [cv2.IMWRITE_JPEG_QUALITY, 95]
                )
                status = (
                    f"Guardada: {filename}"
                    if saved
                    else "Error al guardar la captura"
                )
            else:
                status = "No guardada: no se detectó ningún rostro"

        # Mostrar una copia con el estado; el texto no se añade al archivo.
        preview = anonymized.copy()
        cv2.putText(
            preview,
            status,
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 255, 0),
            2
        )
        cv2.imshow("Vista anonimizada — pulsa q para salir", preview)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()


# Clasifiación de imágenes con OpenCV y Python

Se hace un scanning de las imágenes guardadas en /DataCapture/Raw y se genera un dataset en .csv con las coordenadas detectadas para cada objeto

In [1]:
import csv
import os
from pathlib import Path

import cv2


IMAGE_DIR = Path("../DataCapture/raw")
CSV_OUT = Path("../DataCapture/anotaciones.csv")

EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
WINDOW_NAME = "Anotador de objetos"

# True: empieza en la primera imagen que aún no aparece en el CSV.
# False: comienza desde la primera imagen para permitir revisar todo.
RESUME_AT_FIRST_UNREVIEWED = True


class Annotator:
    def __init__(self, image_paths, csv_path, resume=True):
        self.image_paths = image_paths
        self.csv_path = csv_path
        self.columns = [
            "imagen", "x", "y", "w", "h", "ImageWidth", "ImageHeight"
        ]

        self.index = 0
        self.image = None
        self.image_name = None
        self.current_boxes = []
        self.first_point = None

        # Filas ya existentes y nombres de imágenes con algún registro.
        self.rows = []
        self.recorded_images = set()

        # Cajas recuperadas del CSV, por nombre de imagen.
        self.boxes_by_image = {}

        self.csv_path.parent.mkdir(parents=True, exist_ok=True)
        self.load_csv()

        # Reanudar en la primera imagen sin registro.
        if resume:
            self.index = next(
                (
                    i for i, path in enumerate(self.image_paths)
                    if path.name not in self.recorded_images
                ),
                0,  # Si todas están registradas, permite recorrer desde el inicio.
            )

        self.load_current_image()

    def load_csv(self):
        """Carga los registros y reconstruye las cajas existentes."""
        if not self.csv_path.exists() or self.csv_path.stat().st_size == 0:
            self.rows = []
            return

        with self.csv_path.open(
            "r", newline="", encoding="utf-8-sig"
        ) as f:
            reader = csv.DictReader(f)

            if reader.fieldnames != self.columns:
                raise ValueError(
                    "Las columnas del CSV no coinciden con las esperadas.\n"
                    f"Esperadas: {self.columns}\n"
                    f"Encontradas: {reader.fieldnames}"
                )

            for raw_row in reader:
                row = {
                    col: (raw_row.get(col) or "").strip()
                    for col in self.columns
                }

                image_name = row["imagen"]
                if not image_name:
                    continue

                self.rows.append(row)
                self.recorded_images.add(image_name)

                # Una fila con x, y, w, h numéricos representa una caja.
                try:
                    x = int(float(row["x"]))
                    y = int(float(row["y"]))
                    w = int(float(row["w"]))
                    h = int(float(row["h"]))
                except (ValueError, TypeError):
                    # Por ejemplo: fila vacía que marca imagen sin detecciones.
                    continue

                if w <= 0 or h <= 0:
                    continue

                # El anotador original guardó w/h incluyendo el píxel final.
                x2 = x + w - 1
                y2 = y + h - 1

                self.boxes_by_image.setdefault(image_name, []).append(
                    (x, y, x2, y2)
                )

    def save_csv(self):
        """Reescribe el CSV con los registros actuales."""
        temp_path = self.csv_path.with_suffix(self.csv_path.suffix + ".tmp")

        with temp_path.open("w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=self.columns)
            writer.writeheader()
            writer.writerows(self.rows)

        # Reemplazo para evitar dejar un CSV a medio escribir.
        os.replace(temp_path, self.csv_path)

    def load_current_image(self):
        """Carga la imagen indicada por el índice y sus cajas previas."""
        self.image = None
        self.image_name = None
        self.current_boxes = []
        self.first_point = None

        while self.index < len(self.image_paths):
            path = self.image_paths[self.index]
            image = cv2.imread(str(path))

            if image is not None:
                self.image = image
                self.image_name = path.name
                self.current_boxes = list(
                    self.boxes_by_image.get(self.image_name, [])
                )
                return

            print(f"No se pudo leer; se omitirá: {path.name}")
            self.index += 1

    def on_mouse(self, event, x, y, flags, param):
        """Dos clics izquierdos definen las esquinas de una caja."""
        if event != cv2.EVENT_LBUTTONDOWN or self.image is None:
            return

        height, width = self.image.shape[:2]
        x = max(0, min(int(x), width - 1))
        y = max(0, min(int(y), height - 1))

        if self.first_point is None:
            self.first_point = (x, y)
            return

        x0, y0 = self.first_point
        x1, y1 = min(x0, x), min(y0, y)
        x2, y2 = max(x0, x), max(y0, y)

        if x2 > x1 and y2 > y1:
            box_width = x2 - x1 + 1
            box_height = y2 - y1 + 1

            # Si la imagen tenía una fila vacía de "sin detecciones",
            # se elimina: ahora sí tiene una detección.
            self.rows = [
                row for row in self.rows
                if not (
                    row["imagen"] == self.image_name
                    and not row["x"] and not row["y"]
                    and not row["w"] and not row["h"]
                )
            ]

            self.current_boxes.append((x1, y1, x2, y2))

            row = {
                "imagen": self.image_name,
                "x": x1,
                "y": y1,
                "w": box_width,
                "h": box_height,
                "ImageWidth": width,
                "ImageHeight": height,
            }
            self.rows.append({col: str(row[col]) for col in self.columns})
            self.recorded_images.add(self.image_name)
            self.boxes_by_image[self.image_name] = list(self.current_boxes)

            # Guardado inmediato de cada caja completa.
            self.save_csv()

        self.first_point = None

    def render(self):
        """Dibuja las cajas existentes y nuevas sobre la imagen."""
        display = self.image.copy()

        for x1, y1, x2, y2 in self.current_boxes:
            cv2.rectangle(display, (x1, y1), (x2, y2), (0, 255, 0), 2)

        if self.first_point is not None:
            cv2.drawMarker(
                display,
                self.first_point,
                (0, 0, 255),
                markerType=cv2.MARKER_CROSS,
                markerSize=14,
                thickness=2,
            )

        height = display.shape[0]

        cv2.putText(
            display,
            f"Imagen {self.index + 1}/{len(self.image_paths)}: "
            f"{self.image_name} | Cajas: {len(self.current_boxes)}",
            (10, 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.60,
            (0, 255, 255),
            2,
        )

        cv2.putText(
            display,
            "Dos clics: caja | N: siguiente | P: anterior | Q: salir",
            (10, height - 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 255),
            2,
        )

        return display

    def next_image(self):
        """Avanza y registra una fila vacía solo si nunca se registró."""
        if self.image is None:
            return

        # Solo registra una fila vacía si no hay cajas y no existe ya
        # ningún registro para esta imagen.
        if (
            not self.current_boxes
            and self.image_name not in self.recorded_images
        ):
            height, width = self.image.shape[:2]
            row = {
                "imagen": self.image_name,
                "x": "",
                "y": "",
                "w": "",
                "h": "",
                "ImageWidth": width,
                "ImageHeight": height,
            }
            self.rows.append(row)
            self.recorded_images.add(self.image_name)
            self.save_csv()

        self.index += 1
        self.load_current_image()

    def previous_image(self):
        """Retrocede sin crear registros ni modificar las anotaciones."""
        if self.index > 0:
            self.index -= 1
            self.load_current_image()


def main():
    if not IMAGE_DIR.is_dir():
        raise FileNotFoundError(
            f"No se encontró la carpeta de imágenes: {IMAGE_DIR}"
        )

    image_paths = sorted(
        path for path in IMAGE_DIR.iterdir()
        if path.is_file() and path.suffix.lower() in EXTENSIONS
    )

    if not image_paths:
        raise RuntimeError(
            "No se encontraron imágenes compatibles en la carpeta."
        )

    annotator = Annotator(
        image_paths,
        CSV_OUT,
        resume=RESUME_AT_FIRST_UNREVIEWED,
    )

    if annotator.image is None:
        raise RuntimeError("No se pudo cargar ninguna imagen.")

    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_AUTOSIZE)
    cv2.setMouseCallback(WINDOW_NAME, annotator.on_mouse)

    try:
        while annotator.image is not None:
            cv2.imshow(WINDOW_NAME, annotator.render())
            key = cv2.waitKey(20) & 0xFF

            if key == ord("n"):
                annotator.first_point = None
                annotator.next_image()

            elif key == ord("p"):
                annotator.first_point = None
                annotator.previous_image()

            elif key == ord("q"):
                break

    finally:
        cv2.destroyAllWindows()

    print(f"Registros guardados en el CSV: {len(annotator.rows)}")


if __name__ == "__main__":
    main()


Registros guardados en el CSV: 81


# Generación de muestras
Se generan pequeñas capturas rectángulares de aquellos objetos clasificados en el dataset anterior

In [2]:
import csv
from collections import defaultdict
from pathlib import Path

import cv2


DATA_DIR = Path("..") / "DataCapture"
IMAGE_DIR = DATA_DIR / "raw"
CSV_PATH = DATA_DIR / "anotaciones.csv"
OUTPUT_DIR = DATA_DIR / "NG"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reúne las cajas en el orden en que aparecen en el CSV.
boxes_by_image = defaultdict(list)

with CSV_PATH.open("r", newline="", encoding="utf-8-sig") as csv_file:
    reader = csv.DictReader(csv_file)

    required_columns = {"imagen", "x", "y", "w", "h"}
    if not required_columns.issubset(reader.fieldnames or []):
        raise ValueError(
            f"El CSV debe incluir estas columnas: {sorted(required_columns)}"
        )

    for row_number, row in enumerate(reader, start=2):
        # Omite las filas de imágenes revisadas sin objetos anotados.
        values = [row.get(col, "").strip() for col in ("x", "y", "w", "h")]
        if not all(values):
            continue

        try:
            x, y, w, h = (int(float(value)) for value in values)
        except ValueError:
            print(f"Fila {row_number}: coordenadas inválidas; se omite.")
            continue

        if w <= 0 or h <= 0:
            print(f"Fila {row_number}: ancho o alto no válido; se omite.")
            continue

        # Usa solo el nombre del archivo para buscarlo dentro de raw.
        image_name = Path(row["imagen"]).name
        boxes_by_image[image_name].append((x, y, w, h))


if not boxes_by_image:
    print("No se encontraron cajas válidas en el CSV.")
    raise SystemExit


saved_count = 0

for image_name, boxes in boxes_by_image.items():
    image_path = IMAGE_DIR / image_name
    image = cv2.imread(str(image_path))

    if image is None:
        print(f"No se pudo leer la imagen: {image_path}")
        continue

    image_height, image_width = image.shape[:2]
    source_path = Path(image_name)
    extension = source_path.suffix or ".jpg"

    # Numeración ascendente para cada imagen de origen.
    for number, (x, y, w, h) in enumerate(boxes, start=1):
        # Limita la caja a los bordes de la imagen.
        x1 = max(0, x)
        y1 = max(0, y)
        x2 = min(image_width, x + w)
        y2 = min(image_height, y + h)

        if x2 <= x1 or y2 <= y1:
            print(f"Caja fuera de los límites en {image_name}; se omite.")
            continue

        crop = image[y1:y2, x1:x2]

        output_name = f"{source_path.stem}_{number:03d}{extension}"
        output_path = OUTPUT_DIR / output_name

        if cv2.imwrite(str(output_path), crop):
            print(f"Guardado: {output_path.name}")
            saved_count += 1
        else:
            print(f"No se pudo guardar: {output_path.name}")

print(f"Proceso terminado. Recortes guardados: {saved_count}")


Guardado: captura_20261003_153016_534962_001.jpg
Guardado: captura_20261003_153021_540685_001.jpg
Guardado: captura_20261003_153051_624221_001.jpg
Guardado: captura_20261003_153056_648946_001.jpg
Guardado: captura_20261003_153101_657504_001.jpg
Guardado: captura_20261003_153838_135146_001.jpg
Guardado: captura_20261003_153838_135146_002.jpg
Guardado: captura_20261003_153944_399966_001.jpg
Guardado: captura_20261003_153944_399966_002.jpg
Guardado: captura_20261003_153949_427154_001.jpg
Guardado: captura_20261003_153949_427154_002.jpg
Guardado: captura_20261003_153954_440259_001.jpg
Guardado: captura_20261003_153954_440259_002.jpg
Guardado: captura_20261003_154213_767364_001.jpg
Guardado: captura_20261003_154218_787177_001.jpg
Guardado: captura_20261003_154218_787177_002.jpg
Guardado: captura_20261003_154218_787177_003.jpg
Guardado: captura_20261003_154257_156706_001.jpg
Guardado: captura_20261003_154257_156706_002.jpg
Guardado: captura_20261003_154257_156706_003.jpg
Guardado: captura_20